# Diabetes Prediction using Machine Learning

This notebook builds a machine learning pipeline to predict whether a patient is likely to have diabetes.

It covers:
- dataset loading and exploration
- feature/target separation
- train-test split
- preprocessing
- model training and evaluation
- comparison of multiple classifiers

In [5]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print('Libraries imported successfully.')

Libraries imported successfully.


In [6]:
data_path = Path('diabetes (1).csv')
if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found: {data_path}")

df = pd.read_csv(data_path)
print('Dataset shape:', df.shape)
print('\nFirst few rows:')
display(df.head())
print('\nColumns:', list(df.columns))
print('\nData types:\n', df.dtypes)
print('\nMissing values:\n', df.isnull().sum())
print('\nStatistical summary:\n', df.describe().T)

Dataset shape: (768, 9)

First few rows:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1



Columns: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

Data types:
 Pregnancies                   int64
Glucose                       int64
BloodPressure                 int64
SkinThickness                 int64
Insulin                       int64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                           int64
Outcome                       int64
dtype: object

Missing values:
 Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

Statistical summary:
                           count        mean         std     min       25%  \
Pregnancies               768.0    3.845052    3.369578   0.000   1.00000   
Glucose                   768.0  12

## 1. Data preparation

The dataset contains 768 records and 9 columns. The target variable is `Outcome`, where:
- 0 = No diabetes
- 1 = Diabetes

The dataset is already clean, with no missing values, so no significant imputation is required before model training.

In [7]:
X = df.drop(columns=['Outcome'])
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('y_train distribution:', y_train.value_counts().to_dict())
print('y_test distribution:', y_test.value_counts().to_dict())

X_train shape: (614, 8)
X_test shape: (154, 8)
y_train distribution: {0: 400, 1: 214}
y_test distribution: {0: 100, 1: 54}


## 2. Model building and evaluation

Logistic Regression is a good baseline because it works well for binary classification and is easy to interpret. We also compare it with Random Forest and Gradient Boosting to find the best-performing model.

In [8]:
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_metrics = {
        'accuracy': accuracy_score(y_train, train_pred),
        'precision': precision_score(y_train, train_pred, zero_division=0),
        'recall': recall_score(y_train, train_pred, zero_division=0),
        'f1': f1_score(y_train, train_pred, zero_division=0),
    }

    test_metrics = {
        'accuracy': accuracy_score(y_test, test_pred),
        'precision': precision_score(y_test, test_pred, zero_division=0),
        'recall': recall_score(y_test, test_pred, zero_division=0),
        'f1': f1_score(y_test, test_pred, zero_division=0),
    }

    train_cm = confusion_matrix(y_train, train_pred)
    test_cm = confusion_matrix(y_test, test_pred)

    print(f'\n==== {name} ====')
    print('Training metrics:')
    print({k: round(v, 4) for k, v in train_metrics.items()})
    print('Testing metrics:')
    print({k: round(v, 4) for k, v in test_metrics.items()})
    print('Training confusion matrix:\n', train_cm)
    print('Testing confusion matrix:\n', test_cm)

    results.append({
        'name': name,
        'train_metrics': train_metrics,
        'test_metrics': test_metrics,
        'model': model
    })

best_model = max(results, key=lambda r: r['test_metrics']['f1'])
print('\nBest model based on test F1-score:', best_model['name'])
print('Best test metrics:', {k: round(v, 4) for k, v in best_model['test_metrics'].items()})


==== Logistic Regression ====
Training metrics:
{'accuracy': 0.7915, 'precision': 0.759, 'recall': 0.5888, 'f1': 0.6632}
Testing metrics:
{'accuracy': 0.7143, 'precision': 0.6087, 'recall': 0.5185, 'f1': 0.56}
Training confusion matrix:
 [[360  40]
 [ 88 126]]
Testing confusion matrix:
 [[82 18]
 [26 28]]

==== Random Forest ====
Training metrics:
{'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}
Testing metrics:
{'accuracy': 0.7597, 'precision': 0.6735, 'recall': 0.6111, 'f1': 0.6408}
Training confusion matrix:
 [[400   0]
 [  0 214]]
Testing confusion matrix:
 [[84 16]
 [21 33]]

==== Gradient Boosting ====
Training metrics:
{'accuracy': 0.9186, 'precision': 0.9409, 'recall': 0.8178, 'f1': 0.875}
Testing metrics:
{'accuracy': 0.7532, 'precision': 0.6667, 'recall': 0.5926, 'f1': 0.6275}
Training confusion matrix:
 [[389  11]
 [ 39 175]]
Testing confusion matrix:
 [[84 16]
 [22 32]]

Best model based on test F1-score: Random Forest
Best test metrics: {'accuracy': 0.7597, '

## 3. Interpretation

The dataset is clean and does not require major missing-value treatment. The main preprocessing step is the train/test split, and feature scaling is applied where appropriate for Logistic Regression.

The Random Forest model performs best on this dataset because it captures non-linear relationships in the health features more effectively than a simple linear model.